# Interactive charts with Plotly

Static charts show one view. Interactive charts let a reader hover for exact values, zoom into a
region, and toggle series on and off. This notebook builds interactive figures with Plotly Express
through the module's shared `charts.py` builders.

## Learning objectives

By the end of this notebook you will be able to:

- build line, bar, box, and scatter figures with Plotly Express;
- add hover information and log axes for wide-ranging data;
- animate a figure across a dimension such as year;
- explain when interactivity helps and when it distracts;
- export a figure to a self-contained HTML file.

## Concept

Plotly figures are JavaScript objects described in Python. They render in a notebook, in a
browser, or as a standalone HTML file. Express is the high-level interface: you pass a DataFrame
and column names, and it infers the marks. The shared `charts.py` wraps the common cases so the
theme is consistent across the module.

Interactivity earns its place when the data is dense or multi-dimensional: hovering reveals exact
values, and the legend acts as a filter. It is less useful for a single number, where a static
chart is faster to read. Animations across a dimension such as year turn a two-dimensional chart
into a short film, but they can obscure comparisons if overused.

## Worked example

### Load and import the builders

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import charts
from ds_practice import load_gapminder, set_seed

set_seed(42)
gap = load_gapminder()
world = gap.groupby(["year", "continent"])["lifeExp"].mean().reset_index()
print("rows:", len(gap))

### A line chart

`charts.line_chart` returns a figure; the notebook renders it.

In [ ]:
fig = charts.line_chart(world, x="year", y="lifeExp", color="continent",
                        title="Life expectancy by continent")
fig.show()

### A bar chart

In [ ]:
latest = gap[gap["year"] == gap["year"].max()]
means = latest.groupby("continent", as_index=False)["lifeExp"].mean()
fig = charts.bar_chart(means, x="continent", y="lifeExp", title="Mean life expectancy, 2007")
fig.show()

### A box chart

In [ ]:
fig = charts.box_chart(latest, x="continent", y="lifeExp", color="continent",
                       title="Spread of life expectancy, 2007")
fig.show()

### A bubble chart with hover and a log axis

This is where interactivity shines: hovering names the country and reports income and life
expectancy, and the log axis keeps a wide income range readable.

In [ ]:
fig = charts.bubble_chart(
    latest, x="gdpPercap", y="lifeExp", size="pop", color="continent",
    hover="country", title="GDP per capita vs life expectancy (size = population)",
)
fig.update_xaxes(type="log", title="GDP per capita (USD, log)")
fig.update_yaxes(title="Life expectancy (years)")
fig.show()

### An animated chart

Plotting every five-year period and animating by year shows the whole trajectory in one figure.
Plotly Express adds the play button automatically when `animation_frame` is set.

In [ ]:
import plotly.express as px

animated = px.scatter(
    gap, x="gdpPercap", y="lifeExp", size="pop", color="continent",
    hover_name="country", log_x=True, animation_frame="year", animation_group="country",
    range_x=[100, 100000], range_y=[25, 90], template="plotly_white",
    title="Life expectancy vs GDP, animated by year",
)
animated.show()

### Export to HTML

A Plotly figure can be written to a self-contained file that opens without Python.

In [ ]:
out = Path.cwd() / "gapminder_interactive.html"
animated.write_html(out, include_plotlyjs="cdn")
print("wrote", out.name, "|", out.stat().st_size, "bytes")
out.unlink()
print("removed demo file")

## Exercises

1. **Hover budget.** Rebuild the bubble chart with only `country` and `pop` in the hover. Explain
   why limiting hover fields improves readability.
2. **Small multiples.** Use `charts.line_chart` with `facet_col` on the continent (or build a
   Plotly figure with `facet_col="continent"`) and compare it with the single multi-line chart.
3. **A static alternative.** Pick one interactive chart above and argue in three sentences whether
   the interactivity is essential or merely nice, naming the decision it helps a reader make.

## Limitations

Plotly figures embed a JavaScript library, so exported pages are large; a static PNG may be better
for a report. Animation does not work in every PDF or static export. Hover text cannot be read by
screen readers, so interactive-only charts need an accessible table alongside. Finally, the
convenience of Express hides the underlying figure specification, and deeply customised charts
eventually require the lower-level `graph_objects` API.